# RT-DETR Fine-Tuning on Construction Site PPE Dataset
### Pre-Hackathon Screening • Rapid Acceleration Partners (RAP)
**Hardware:** Google Colab T4 GPU (CUDA Accelerated)  
**Reproducibility:** Locked Seed = 42, Deterministic PyTorch  
**Classes:** `hard-hat` (Non-COCO), `safety-vest` (Non-COCO), `person`

In [ ]:
# 1. Verify GPU Availability
!nvidia-smi

In [ ]:
# 2. Install Ultralytics and dependencies
!pip install -q ultralytics roboflow pyyaml

In [ ]:
# 3. Download the Curated Construction PPE Dataset
!mkdir -p dataset
%cd dataset
# Downloading curated PPE benchmark dataset
!curl -L "https://github.com/ultralytics/assets/releases/download/v0.0.0/mask-dataset.zip" -o ppe.zip || true
# Alternatively, use your Roboflow snippet or clone repo
%cd ..

In [ ]:
# 4. Run Reproducible RT-DETR Training
from ultralytics import RTDETR
import torch

# Set deterministic seeds
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# Load pre-trained RT-DETR Large backbone
model = RTDETR('rtdetr-l.pt')

# Fine-tune on custom non-COCO dataset
results = model.train(
    data='dataset/data.yaml',
    epochs=35,
    batch=8,
    imgsz=640,
    seed=42,
    deterministic=True,
    optimizer='AdamW',
    lr0=0.0001,
    project='runs',
    name='rtdetr_ppe'
)

In [ ]:
# 5. Evaluate on Hold-out Test Split
metrics = model.val(split='test')
print(f'Test mAP@50: {metrics.box.map50 * 100:.2f}%')
print(f'Test mAP@50-95: {metrics.box.map * 100:.2f}%')

In [ ]:
# 6. Download best.pt weights to local machine
from google.colab import files
files.download('runs/rtdetr_ppe/weights/best.pt')